In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shrutimechlearn/churn-modelling/Churn_Modelling.csv


In [15]:
df = pd.read_csv('/kaggle/input/datasets/shrutimechlearn/churn-modelling/Churn_Modelling.csv')

In [16]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [17]:
df = df.drop(columns = ['RowNumber' , 'CustomerId' , 'Surname'])

In [18]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [41]:
from sklearn.model_selection import train_test_split , cross_val_score , StratifiedKFold

In [20]:
X = df.drop('Exited' , axis = 1)
y = df['Exited']

In [21]:
X_train,X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [22]:
X_train['Geography'].value_counts()

Geography
France     3994
Germany    2011
Spain      1995
Name: count, dtype: int64

In [23]:
X_train.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
9254,686,France,Male,32,6,0.00,2,1,1,179093.26
1561,632,Germany,Male,42,4,119624.60,2,1,1,195978.86
1670,559,Spain,Male,24,3,114739.92,1,1,0,85891.02
6087,561,France,Female,27,9,135637.00,1,1,0,153080.40
6669,517,France,Male,56,9,142147.32,1,0,0,39488.04


In [24]:
def createIsZero(X):
    x = X.copy()
    x['isZeroBalance'] = (x['Balance']==0.00).astype(int)
    return x

In [25]:
from sklearn.preprocessing import OneHotEncoder , FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [26]:
preprocesser = ColumnTransformer(transformers = [
    ('ohe' , OneHotEncoder(sparse_output = False) , ['Geography' , 'Gender']),
    ('func_std' , Pipeline(
        steps = [
            ('log', FunctionTransformer(func = np.log1p , inverse_func = np.expm1)), 
            ('scale' , StandardScaler())
        ]
    ) , ['Balance' , 'Age' , 'CreditScore' , 'EstimatedSalary'])
] , remainder = 'passthrough')

In [27]:
full_preprocessing = Pipeline(steps = [
    ('Add_is_zero_balance' , FunctionTransformer(createIsZero)) , 
    ('preprocessing' , preprocesser)
])

In [28]:
from sklearn.ensemble import StackingClassifier

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
import xgboost as xgb

In [29]:
imbalanced_ratio = 1607/393

In [30]:
!pip install optuna

In [58]:
import optuna

def objective(trial):
    rf__n_estimators = trial.suggest_int('rf_n_estimators' , 50 , 200)
    rf__max_depth = trial.suggest_int('rf_max_depth' , 3 , 7)

    svc__C = trial.suggest_float('svc_C' , 0.1 , 10.0 , log = True)

    xgboost__learning_rate = trial.suggest_float('xgboost_learning_rate' ,0.05 , 0.1)
    xgboost__n_estimators = trial.suggest_int('xgboost_n_estimators' ,50, 200)
    xgboost__max_depth = trial.suggest_int('xgboost_max_depth' , 3 , 7)

    base_estimators = [
        ('rf' , RandomForestClassifier(n_estimators = rf__n_estimators , criterion = 'gini' , max_depth = rf__max_depth , n_jobs = -1 , class_weight = 'balanced')),
        ('svc' , SVC(kernel = 'rbf' , random_state = 42 , probability=True , class_weight = 'balanced' , C = svc__C)),
        ('xgboost' ,xgb.XGBClassifier(eta = 0.1 ,learning_rate = xgboost__learning_rate  ,  n_estimators = xgboost__n_estimators , random_state = 42 , max_depth= xgboost__max_depth , scale_pos_weight = imbalanced_ratio))
    ]
    
    meta_estimators = LogisticRegression(random_state = 42 , class_weight = 'balanced' , C = 1.0)
    
    training_model = StackingClassifier(
        estimators = base_estimators ,
        final_estimator = meta_estimators , 
        cv = 3,
        n_jobs = -1,
        stack_method = 'predict_proba'
    )

    final_pipeline = Pipeline(steps = [
        ('preprocessing' , full_preprocessing) ,
        ('model' , training_model)
    ])

    cv = StratifiedKFold(n_splits = 5 , shuffle = True , random_state = 42)
    scores = cross_val_score(final_pipeline,X_train , y_train , cv= cv , scoring = 'roc_auc', n_jobs = -1)

    return scores.mean()
    


In [59]:
from optuna.samplers import TPESampler

In [60]:
study = optuna.create_study(
    direction = 'maximize',
    sampler = TPESampler(seed = 42)
)

[I 2026-08-14 09:10:07,787] A new study created in memory with name: no-name-51a54297-f46b-42f2-b7e6-229464d37162


In [61]:
study.optimize(objective, n_trials=30, show_progress_bar=True)

  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-08-14 09:11:29,028] Trial 0 finished with value: 0.8638598490457896 and parameters: {'rf_n_estimators': 106, 'rf_max_depth': 7, 'svc_C': 2.9106359131330697, 'xgboost_learning_rate': 0.07993292420985183, 'xgboost_n_estimators': 73, 'xgboost_max_depth': 3}. Best is trial 0 with value: 0.8638598490457896.
[I 2026-08-14 09:12:49,398] Trial 1 finished with value: 0.861231675132346 and parameters: {'rf_n_estimators': 58, 'rf_max_depth': 7, 'svc_C': 1.5930522616241019, 'xgboost_learning_rate': 0.08540362888980227, 'xgboost_n_estimators': 53, 'xgboost_max_depth': 7}. Best is trial 0 with value: 0.8638598490457896.
[I 2026-08-14 09:14:08,313] Trial 2 finished with value: 0.8632404068851154 and parameters: {'rf_n_estimators': 175, 'rf_max_depth': 4, 'svc_C': 0.23102018878452935, 'xgboost_learning_rate': 0.059170225492671695, 'xgboost_n_estimators': 95, 'xgboost_max_depth': 5}. Best is trial 0 with value: 0.8638598490457896.
[I 2026-08-14 09:15:18,755] Trial 3 finished with value: 0.86323

In [64]:
study.best_value

0.8651372860415965

In [71]:
for key , value in study.best_params.items():
    print(f" {key} : {value}")

 rf_n_estimators : 170
 rf_max_depth : 6
 svc_C : 0.7558543394003551
 xgboost_learning_rate : 0.07053532581402962
 xgboost_n_estimators : 146
 xgboost_max_depth : 3


In [72]:
best_p = study.best_params

In [73]:
best_p

{'rf_n_estimators': 170,
 'rf_max_depth': 6,
 'svc_C': 0.7558543394003551,
 'xgboost_learning_rate': 0.07053532581402962,
 'xgboost_n_estimators': 146,
 'xgboost_max_depth': 3}

In [84]:
best_base_estimators = [
    ('rf' , RandomForestClassifier(n_estimators = best_p['rf_n_estimators'] , criterion = 'gini' , max_depth = best_p['rf_max_depth'] , n_jobs = -1 , class_weight = 'balanced')),
    ('svc' , SVC(kernel = 'rbf' , random_state = 42 , probability=True , class_weight = 'balanced' , C = best_p['svc_C'])),
    ('xgboost',xgb.XGBClassifier(eta = 0.1 ,learning_rate = best_p['xgboost_learning_rate']  ,  n_estimators = best_p['xgboost_n_estimators'] , random_state = 42 , max_depth= best_p['xgboost_max_depth'] , scale_pos_weight = imbalanced_ratio))
]

In [85]:
stacking_final = StackingClassifier(
    estimators = best_base_estimators,
    final_estimator = LogisticRegression(random_state = 42 , class_weight = 'balanced' , C = 1.0),
    cv = 5 ,
    stack_method = 'predict_proba',
    n_jobs = -1    
)

In [86]:
final_pipeline = Pipeline(
    steps = [
        ('prep' , full_preprocessing),
        ('model' , stacking_final)
    ]
)

In [87]:
final_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('prep',
                 Pipeline(steps=[('Add_is_zero_balance',
                                  FunctionTransformer(func=<function createIsZero at 0x7c33cdd5e020>)),
                                 ('preprocessing',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('ohe',
                                                                   OneHotEncoder(sparse_output=False),
                                                                   ['Geography',
                                                                    'Gender']),
                                                                  ('func_std',
                                                                   Pipeline(steps=[('log',
                                                                                    FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                                        inverse...
                                                               learning_rate=0.07053532581402962,
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=3,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=146,
                                                               n_jobs=None, ...))],
                                    final_estimator=LogisticRegression(class_weight='balanced',
                                                                       random_state=42),
                                    n_jobs=-1, stack_method='predict_proba'))])

In [104]:
from sklearn.metrics import accuracy_score , confusion_matrix , precision_score , recall_score , classification_report , roc_auc_score

In [122]:
for thresh in [0.52 , 0.55 , 0.57 , 0.59  , 0.60]:
    y_proba = final_pipeline.predict_proba(X_test)[: , 1]
    y_pred_custom = (y_proba >= thresh).astype(int)

    print(f" {thresh} : {classification_report(y_test , y_pred_custom)}")

 0.52 :               precision    recall  f1-score   support

           0       0.94      0.81      0.87      1607
           1       0.50      0.78      0.61       393

    accuracy                           0.80      2000
   macro avg       0.72      0.79      0.74      2000
weighted avg       0.85      0.80      0.82      2000

 0.55 :               precision    recall  f1-score   support

           0       0.94      0.83      0.88      1607
           1       0.52      0.77      0.62       393

    accuracy                           0.82      2000
   macro avg       0.73      0.80      0.75      2000
weighted avg       0.85      0.82      0.83      2000

 0.57 :               precision    recall  f1-score   support

           0       0.94      0.84      0.89      1607
           1       0.54      0.76      0.63       393

    accuracy                           0.83      2000
   macro avg       0.74      0.80      0.76      2000
weighted avg       0.86      0.83      0.84      2

In [123]:
y_proba = final_pipeline.predict_proba(X_test)[: , 1]

y_pred_custom = (y_proba >= 0.60).astype(int)

In [124]:
print(classification_report(y_test , y_pred_custom))

              precision    recall  f1-score   support

           0       0.93      0.86      0.90      1607
           1       0.57      0.74      0.64       393

    accuracy                           0.84      2000
   macro avg       0.75      0.80      0.77      2000
weighted avg       0.86      0.84      0.85      2000

